In [1]:
%%capture
# 필요한 패키지 설치
%pip install "langchain>=0.3.0" "langchain-openai>=0.2.0" "langchain-upstage>=0.3.0" "langgraph>=0.2.0" "python-dotenv>=1.0.0" -q 

In [2]:
import os
import warnings
from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# .env 파일에서 환경변수 로드
load_dotenv()

# API 키 확인 (.env 미설정 시 직접 입력)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = input("🔑 OPENAI_API_KEY를 입력하세요: ")

if not os.environ.get("UPSTAGE_API_KEY"):
    os.environ["UPSTAGE_API_KEY"] = input("🔑 UPSTAGE_API_KEY를 입력하세요: ")
print("✅ 환경 설정 완료") 

🔑 OPENAI_API_KEY를 입력하세요:  


✅ 환경 설정 완료


In [4]:
from langchain_openai import ChatOpenAI

# ═══════════════════════════════════════════════════════════════
# 모델 초기화 — OpenAI / Solar 중 택 1 (주석 해제하여 전환)
# ═══════════════════════════════════════════════════════════════

# # ── Option A: OpenAI (기본) ──
# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Option B: Solar 대안 (주석 해제하여 사용) ──
from langchain_upstage import ChatUpstage
llm = ChatUpstage(model="solar-pro3")

In [7]:
# 모델 연결 테스트
response = llm.invoke("3일 서울 여행 계획을 간단히 요약해줘.")
print("LLM 응답:")
print(response.content[:200] + "...")
print(f"\n✅ 모델 연결 확인 완료 ({llm.model_name})") 

LLM 응답:
**3일 서울 여행 핵심 요약**

| Day | 주요 일정 (추천) | 비고 |
|---|---|---|
| **Day 1** | **오전** – 인천국제공항 도착 → 공항버스/지하철로 숙소 이동 <br>**오후** – 경복궁·청와대 관람, 전통 한복 체험 <br>**저녁** – 인사동·남대문 야시장 탐방 | 날씨가 좋다면 청계천 산책도 가능 |
| **...

✅ 모델 연결 확인 완료 (solar-pro3)


In [8]:
# TODO: 단일 Agent로 복잡한 작업 시도

# 복잡한 작업 요청
complex_request = """
3일 서울 여행 계획을 세워줘.
- 각 날짜별 상세 일정
- 점심, 저녁 맛집 추천 (위치, 메뉴, 가격대)
- 관광지 추천 (입장료, 소요시간)
- 전체 예산 계산
"""

# TODO: llm.invoke()를 사용하여 단일 Agent로 처리하세요
response = llm.invoke(complex_request)

if response is not None:
    print("=== 단일 Agent 응답 ===")
    print(response.content)
else:
    print("=== 코드를 완성해주세요 ===") 

=== 단일 Agent 응답 ===
아래는 **2025년 12월 25일(목), 26일(금), 27일(토)**을 예시로 한 **3일 서울 여행 계획**입니다.  
(날짜는 필요에 따라 조정하시고, 2026년 3월 31일 기준으로 최신 정보를 반영했습니다.)

---

## 1️⃣ 여행 개요
| 항목 | 내용 |
|------|------|
| **여행 기간** | 2박 3일 (예시: 12/25 ~ 12/27) |
| **숙소** | **서울역 앞 ‘라마다스위트’(Ramada Suites Seoul Station)** – 1박 120,000 원, 조식 포함, KTX·지하철 접근이 편리 |
| **교통** | **서울 메트로 1일 패스** (성인 8,000 원) – 3일 모두 사용 가능 (첫 날 1일 패스 구매 후 재사용) |
| **예산 (예시)** | 숙박 2박 ≈ 240,000 원<br>교통 1일 패스 × 3 ≈ 24,000 원<br>점심 × 3 ≈ 15,000 원<br>저녁 × 3 ≈ 30,000 원<br>입장료 ≈ 15,000 원<br>기타·기념품 ≈ 10,000 원<br>**총합 ≈ 334,000 원** (1인당) |

> **※ 실제 비용은 선택한 식당·숙소·교통 수단에 따라 차이가 날 수 있습니다.** 아래 일정은 비용 절감을 위해 **점심·저녁 모두 15,000 원~30,000 원** 정도의 가성비 좋은 맛집을 중심으로 구성했습니다.

---

## 2️⃣ 일별 상세 일정

### 📅 1일차 – 12월 25일 (목) – “역사·문화 탐험”

| 시간 | 일정 | 비고 |
|------|------|------|
| **08:30** | 숙소 → **경복궁** (서울 종로구) | 도보 10분, 입장료 3,000 원 (성인) |
| **09:00~10:30** | 경복궁 관람 (근정전, 광화문, 국립고궁박물관) | 1시간 30분 소요 |
| **10:45** | 경복궁 → **삼청동** (산책) | 거리 1km, 15분 도보 |
| **11:30~12:

# Multi-Agent

In [9]:
# LangGraph import 및 State 정의
from langgraph.graph import StateGraph, END
from typing import TypedDict, List

class AgentState(TypedDict):
    """Multi-Agent 시스템의 상태 정의"""
    user_request: str     # 사용자 요청
    plan: str             # Planner가 생성한 계획
    result: str           # Worker가 생성한 결과
    reflection: str       # Reflection Agent의 검토 결과
    final_result: str     # 최종 결과
    reflection_count: int # Reflection 반복 횟수 (무한 루프 방지)
    messages: List[dict]  # 에이전트 간 메시지 기록

# State 구조 확인
print("AgentState 필드:")
print("  - user_request: 사용자 요청 문자열")
print("  - plan: Planner Agent의 계획")
print("  - result: Worker Agent의 실행 결과")
print("  - reflection: Reflection Agent의 검토 결과")
print("  - final_result: 최종 결과")
print("  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)")
print("  - messages: 에이전트 간 메시지 기록") 

AgentState 필드:
  - user_request: 사용자 요청 문자열
  - plan: Planner Agent의 계획
  - result: Worker Agent의 실행 결과
  - reflection: Reflection Agent의 검토 결과
  - final_result: 최종 결과
  - reflection_count: Reflection 반복 횟수 (무한 루프 방지)
  - messages: 에이전트 간 메시지 기록


## Planner - Worker

In [10]:
# TODO: Planner Node 구현

def planner_node(state: AgentState) -> AgentState:
    """
    Planner Node: 사용자 요청을 분석하고 단계별 계획을 수립합니다.
    
    Args:
        state: 현재 AgentState
    
    Returns:
        업데이트된 AgentState (plan 필드 추가)
    """
    user_request = state["user_request"]
    
    # TODO: Planner 프롬프트를 작성하세요
    # - 사용자 요청을 분석하고 단계별 계획을 수립하도록 지시
    # - "계획만 수립하고, 직접 실행하지 말 것"을 명시
    # - 출력 형식: Step 1: [작업], Step 2: [작업], ...
    planner_prompt = f"""
        당신은 작업 계획을 수립하는 Planner Agent입니다.
    
        사용자 요청을 분석하고, 수행해야 할 작업을 단계별로 나열하세요.
        각 단계는 구체적이고 실행 가능해야 합니다.
        
        중요: 계획만 수립하세요. 직접 실행하지 마세요.
        
        출력 형식:
        Step 1: [작업 설명]
        Step 2: [작업 설명]
        ...
        
        ---
        사용자 요청: {user_request}
    """
    
    # LLM 호출
    response = llm.invoke(planner_prompt)
    plan = response.content
    
    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Planner",
        "receiver": "Worker",
        "content": plan,
        "task_type": "plan"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "plan": plan,  # TODO: plan 변수로 교체
        "messages": messages
    }

# 테스트
test_state = {
    "user_request": "3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산 포함.",
    "plan": "",
    "result": "",
    "messages": []
}

# TODO 완료 후 테스트
result_state = planner_node(test_state)
if result_state['plan'] is not None:
    print("=== Planner Node 결과 ===")
    print(f"계획:\n{result_state['plan']}")
else:
    print("=== 코드를 완성해주세요 ===")

=== Planner Node 결과 ===
계획:
Step 1: 여행 일정 초안 작성  
- 여행 시작일, 종료일, 이동 수단, 숙박 장소 등을 포함한 전체 일정을 3일 기준으로 계획  

Step 2: 관광지 선정 및 방문 순서 결정  
- 첫날: 인사동, 경복궁, 북촌한옥마을 등  
- 둘째날: 남산타워, 동대문디자인플라자, 한강공원  
- 셋째날: 홍대, 서울숲, 남산골한옥마을  

Step 3: 맛집 조사 및 추천  
- 각 지역별 대표 맛집(예: 인사동 전통 한정식, 홍대 퓨전 음식, 동대문 길거리 음식) 및 예산(1인기준, 평균 10,000~15,000원) 정리  

Step 4: 예산 산정 및 관리 방안 설정  
- 교통비, 숙박비, 식사비, 기타 비용(관광 입장료 등) 포함 전체 예산 계산  
- 예산 범위 내/초과 방지 위한 체크리스트 작성  

Step 5: 최종 일정표 및 상세 정보 정리  
- 시간별 일정, 이동 방법, 예상 소요 시간, 연락처, 예약 필요 여부 등을 포함한 최종 여행 플래너 작성  

Step 6: 사용자 검토 요청 및 수정 반영 준비  
- 작성된 계획을 사용자에게 공유하고 피드백 받아 최종 확정 전까지 수정 가능 상태 유지


In [11]:

def worker_node(state: AgentState) -> AgentState:
    """
    Worker Node: Planner의 계획을 받아 각 단계를 실행합니다.
    
    Args:
        state: 현재 AgentState (plan 포함)
    
    Returns:
        업데이트된 AgentState (result 필드 추가)
    """
    plan = state["plan"]
    user_request = state["user_request"]
    
    # TODO: Worker 프롬프트를 작성하세요
    # - Planner의 계획을 받아 각 단계를 실행하도록 지시
    # - 각 단계별 구체적인 정보를 포함하도록 요청
    worker_prompt = f"""
        당신은 계획을 실행하는 Worker Agent입니다.
        
        Planner가 수립한 계획을 받아 각 단계를 실행하고 결과를 제공하세요.
        각 단계별로 구체적인 정보를 포함하세요.
        
        ---
        원본 요청: {user_request}
        
        실행할 계획:
        {plan}
        
        ---
        위 계획을 단계별로 실행하고, 각 단계의 결과를 자세히 작성하세요.
    """
    
    # LLM 호출
    response = llm.invoke(worker_prompt)
    result = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Worker",
        "receiver": "User",
        "content": result,
        "task_type": "result"
    })
    
    # TODO: State 업데이트하여 반환
    return {
        **state,
        "result": result,  # TODO: result 변수로 교체
        "messages": messages
    }

# TODO 완료 후 테스트
final_state = worker_node(result_state)
if final_state['result'] is not None:
    print("=== Worker Node 결과 ===")
    print(f"실행 결과:\n{final_state['result'][:500]}...")
else:
    print("=== 코드를 완성해주세요 ===") 

=== Worker Node 결과 ===
실행 결과:
**Worker Agent 실행 결과 보고서**  

**원본 요청**: 3일 서울 여행 계획을 세워줘. 맛집, 관광지, 예산 포함.  

**Planner가 수립한 실행 계획**:  
1. 여행 일정 초안 작성  
2. 관광지 선정 및 방문 순서 결정  
3. 맛집 조사 및 추천  
4. 예산 산정 및 관리 방안 설정  
5. 최종 일정표 및 상세 정보 정리  
6. 사용자 검토 요청 및 수정 반영 준비  

---

## ✅ Step 1: 여행 일정 초안 작성  

| 항목 | 내용 |
|------|------|
| **여행 기간** | 2026‑04‑01 (수) ~ 2026‑04‑03 (금) – 3일간 |
| **이동 수단** | - 첫날: 공항 → 서울 (공항철도 + 지하철) <br> - 이후: 지하철 2~3회 이용, 도보 이동 위주 |
| **숙박 장소** | **서울 시내 중심가** – 2박 (Airbnb 또는 호텔) <br> - 예시: 강남구 신사동 ‘라마다 프라자 호...


In [15]:
from langgraph.graph import StateGraph, START, END

# LangGraph 워크플로우 정의 (Guided Build - 코드 제공)
workflow = StateGraph(AgentState)

# 노드 추가
workflow.add_node("planner", planner_node)
workflow.add_node("worker", worker_node)

# 엣지 연결: planner → worker → END
workflow.add_edge(START, "planner")
workflow.add_edge("planner", "worker")
workflow.add_edge("worker", END)

# 컴파일
app = workflow.compile()

print("LangGraph 워크플로우 구성 완료!")
print("흐름: planner → worker → END") 

LangGraph 워크플로우 구성 완료!
흐름: planner → worker → END


In [17]:
# LangGraph 워크플로우 실행
def run_multi_agent(user_request: str) -> dict:
    """
    LangGraph Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 LangGraph Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    
    final_state = app.invoke(initial_state)
    
    print("✅ 실행 완료\n")
    
    return final_state

# 테스트 실행
workflow_result = run_multi_agent("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 최종 결과")
print("=" * 50)
print(workflow_result["result"]) 

🚀 LangGraph Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
✅ 실행 완료

📊 최종 결과
We need to follow the instruction: "당신은 계획을 실행하는 Worker Agent입니다. Planner가 수립한 계획을 받아 각 단계를 실행하고 결과를 제공하세요. 각 단계별로 구체적인 정보를 포함하세요."

We are given a plan with steps 1-10. We need to execute each step and provide results. However we lack user details: companion number, travel style, special interests. We should probably ask for missing details before proceeding with steps 2-10. But the instruction says we need to execute the plan and provide results. Possibly we can assume generic 2-person, typical interests. But we might need to note missing info and ask for clarification. The system says we need to execute each step and provide detailed info.

Thus we can present each step's execution: step 1 result: we analyzed request, identified missing info, ask user. Then we can proceed with step 2-10 using assumptions, but clearly state assumptions. Provide budget allocation, accommodation o

## Reflection

In [18]:
# TODO: Reflection Node 구현

MAX_REFLECTIONS = 1  # 최대 Reflection 반복 횟수 (무한 루프 방지)

def reflection_node(state: AgentState) -> AgentState:
    """
    Reflection Node: Worker의 결과를 검토하고 개선 제안을 합니다.
    
    Args:
        state: 현재 AgentState (result 포함)
    
    Returns:
        업데이트된 AgentState (reflection, final_result 필드 추가)
    """
    result = state["result"]
    user_request = state["user_request"]
    plan = state["plan"]
    
    # TODO: Reflection 프롬프트를 작성하세요
    # - Worker의 결과를 검토하고 원본 요청 충족 여부 평가
    # - 검토 항목: 요구사항 충족, 누락 정보, 정확성, 개선 필요 사항
    # - 출력 형식: ## 검토 결과 + ## 최종 개선 결과
    reflection_prompt = f"""
        당신은 결과를 검토하는 Reflection Agent입니다.
        
        Worker Agent가 생성한 결과를 검토하고, 원본 요청을 충족하는지 평가하세요.
        
        검토 항목:
        1. 원본 요청의 모든 요구사항이 충족되었는가?
        2. 누락된 정보가 있는가?
        3. 정보의 정확성과 구체성은 적절한가?
        4. 개선이 필요한 부분이 있는가?
        
        ---
        원본 요청: {user_request}
        
        실행된 계획:
        {plan}
        
        Worker의 결과:
        {result}
        
        ---
        위 내용을 검토하고 다음 형식으로 작성하세요:
        
        ## 검토 결과
        - 충족 여부: [충족/부분 충족/미충족]
        - 강점: [잘된 부분]
        - 개선 필요: [부족한 부분]
        
        ## 최종 개선 결과
        [Worker의 결과를 보완하여 최종 결과 작성]
    """
    
    # LLM 호출
    response = llm.invoke(reflection_prompt)
    reflection = response.content

    # 메시지 기록 추가
    messages = state.get("messages", [])
    messages.append({
        "sender": "Reflection",
        "receiver": "User",
        "content": reflection,
        "task_type": "reflection"
    })
    
    # TODO: State 업데이트하여 반환 (reflection_count 증가 포함)
    return {
        **state,
        "reflection": reflection,  # TODO: reflection 변수로 교체
        "final_result": reflection,  # TODO: reflection 변수로 교체
        "reflection_count": state.get("reflection_count", 0) + 1,
        "messages": messages
    }

def should_continue_reflection(state: AgentState) -> str:
    """Reflection 후 종료할지 Worker로 돌아갈지 판단"""
    count = state.get("reflection_count", 0)
    reflection = state.get("reflection", "")
    
    # 최대 반복 횟수 초과 시 종료
    if count >= MAX_REFLECTIONS:
        return "end"
    
    # "충족" 판정이면 종료, 아니면 Worker로 재실행
    if "충족 여부: 충족" in reflection:
        return "end"
    
    return "worker"

# TODO 완료 후 테스트
reflection_state = reflection_node(final_state)
if reflection_state['reflection'] is not None:
    print("=== Reflection Node 결과 ===")
    print(f"검토 결과:\n{reflection_state['reflection'][:800]}...")
else:
    print("=== 코드를 완성해주세요 ===") 

=== Reflection Node 결과 ===
검토 결과:
## 검토 결과
- 충족 여부: **부분 충족**
- 강점:  
  - 일정 구성이 3일 동안의 **주요 관광지**(인사동, 경복궁, 북촌한옥마을, 남산타워, DDP, 한강공원, 홍대, 서울숲, 남산골한옥마을)를 골고루 포함하고 있음  
  - **예산 항목**(숙박, 교통, 식비, 입장료, 기타)을 상세히 제시하고, 체크리스트를 통해 예산 관리 방안을 제시함  
  - 각 관광지와 맛집에 **전화번호, 예약 필요 여부** 등 실용적인 정보를 제공함  
  - 시간대별 **시간표**와 이동 방법·소요시간을 명시해 실제 여행 실행에 도움이 됨  

- 개선 필요:  
  1. **예산 정확성**: 제시된 총 예산(₩400,000)과 초기 예산(₩31만원) 사이에 큰 차이가 있음. 구체적인 비용 산정 근거가 부족해 사용자가 실제 예산을 예측하기 어려움.  
  2. **숙소 위치**: 인사동·북촌·남산 등 주요 관광지와 거리가 다소 멀어 이동 시간이 많이 소요됨. 중심가(한옥스테이 인사동 등)로 숙소 변경을 고려할 필요가 있음.  
  3. **맛집 상세 정보 부족**: 메뉴 가격 외에 영업시간, 휴무일, 인기 메뉴 등 구체적인 정보가 부족함.  
  4. **예산 관리 체크리스트 구체화**: “매일 지출 기록”이라는 항목은 좋은데, 실제 기록 방법(예: 엑셀, 노트, 앱) 및 초과 시 대처방안을 구체화하면 더 실용적임.  
  5. **비상 상황 대비**: 날씨 변동, 교통 지연, 식당 대기 시간 등에 대한 **대체 플랜**이 제시되지 않음.  

---  

## 최종 개선 결과  

### 1️⃣ 예산 산정...


In [19]:
# TODO: 성과 평가 함수 구현
# 아래 함수를 완성하세요.

import json
import re

def evaluate_multi_agent(state: AgentState) -> dict:
    """
    Multi-Agent 시스템의 성과를 평가합니다.
    
    Args:
        state: 현재 AgentState (user_request, plan, result, reflection 포함)
    
    Returns:
        dict: {"score": int, "feedback": str, "details": dict}
    """

    # 평가 프롬프트 구성
    eval_prompt = f"""
        당신은 Multi-Agent 시스템의 성과를 평가하는 평가자입니다.
        
        ## 평가 기준
        1. Planner 평가 (0~30점): 계획의 완성도, 단계 구성의 적절성
        2. Worker 평가 (0~40점): 실행 품질, 정보의 충실도와 정확성
        3. 전체 평가 (0~30점): 사용자 요청 충족도, 협업 효율성
        
        ## 입력 정보
        - 사용자 요청: {state['user_request']}
        - Planner의 계획: {state['plan']}
        - Worker의 결과: {state['result']}
        - Reflection 검토: {state.get('reflection', '없음')}
        
        ## 출력 형식
        반드시 아래 JSON 형식만 출력하세요. 다른 텍스트는 포함하지 마세요.
        {{"planner_score": <0~30>, "worker_score": <0~40>, "overall_score": <0~30>, "feedback": "<전체 피드백>"}}
    """

    response = llm.invoke(eval_prompt)
    content = response.content

    # JSON 파싱 (마크다운 코드 블록 처리)
    try:
        # 마크다운 코드 블록에서 JSON 추출 (```json ... ``` 또는 ``` ... ```)
        json_match = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', content)
        if json_match:
            json_str = json_match.group(1)
        else:
            # 코드 블록이 없으면 중괄호로 시작하는 JSON 추출
            json_match = re.search(r'\{[\s\S]*\}', content)
            json_str = json_match.group(0) if json_match else content

        result = json.loads(json_str)

        # TODO: result를 이용하여 점수를 합산하고 결과를 반영하세요
        total_score = result['planner_score'] + result['worker_score'] + result['overall_score']      # TODO: 세 항목의 점수를 합산하세요

        return {
            "score": total_score,
            "feedback": result['feedback'],       # TODO: result의 feedback 값으로 교체하세요
            "details": {
                "planner": result['planner_score'],    # TODO: result의 planner_score 값으로 교체하세요
                "worker": result['worker_score'],     # TODO: result의 worker_score 값으로 교체하세요
                "overall": result['overall_score']     # TODO: result의 overall_score 값으로 교체하세요
            }
        }
    except Exception as e:
        return {"score": 0, "feedback": f"평가 실패: {str(e)}", "details": {}}

# 테스트
evaluation = evaluate_multi_agent(reflection_state)
if evaluation['feedback'] is not None:
    print("=== 성과 평가 결과 ===")
    print(f"종합 점수: {evaluation['score']}/100")
    print(f"세부 점수: Planner {evaluation['details'].get('planner', 0)}/30, Worker {evaluation['details'].get('worker', 0)}/40, 전체 {evaluation['details'].get('overall', 0)}/30")
    print(f"피드백: {evaluation['feedback']}")
else:
    print("=== 코드를 완성해주세요 ===") 

=== 성과 평가 결과 ===
종합 점수: 80/100
세부 점수: Planner 25/30, Worker 35/40, 전체 20/30
피드백: 계획은 관광지 선정과 예산 관리 측면에서 기본적인 틀을 잘 구성했으나, 예산 산정의 정확도와 숙소 위치 최적화, 맛집 상세 정보, 대체 플랜 제시 등에서 개선이 필요합니다. 예산 초과 방지를 위한 구체적인 실행 방안과 사용자 맞춤형 조정 가능성을 높이는 것이 핵심입니다.


In [20]:
# Reflection 패턴 워크플로우 구성
workflow_reflection = StateGraph(AgentState)

# 노드 추가
workflow_reflection.add_node("planner", planner_node)
workflow_reflection.add_node("worker", worker_node)
workflow_reflection.add_node("reflection", reflection_node)

# 엣지 연결: planner → worker → reflection
workflow_reflection.add_edge(START, "planner")
workflow_reflection.add_edge("planner", "worker")
workflow_reflection.add_edge("worker", "reflection")

# Reflection 후 조건 분기: END 또는 Worker 재실행
workflow_reflection.add_conditional_edges(
    "reflection",
    should_continue_reflection,
    {"end": END, "worker": "worker"}
)

# 컴파일
app_reflection = workflow_reflection.compile()

print("Reflection 패턴 워크플로우 구성 완료!")
print("흐름: planner → worker → reflection → (조건 분기) → END 또는 worker") 

Reflection 패턴 워크플로우 구성 완료!
흐름: planner → worker → reflection → (조건 분기) → END 또는 worker


In [21]:
# Reflection 패턴 워크플로우 실행
def run_multi_agent_reflection(user_request: str) -> dict:
    """
    Reflection 패턴이 포함된 Multi-Agent 시스템 실행
    
    Args:
        user_request: 사용자 요청
    
    Returns:
        최종 상태를 담은 딕셔너리
    """
    print("🚀 Reflection Multi-Agent 시스템 시작\n")
    
    # 초기 상태 설정
    initial_state = {
        "user_request": user_request
    }
    
    # 워크플로우 실행
    print("📋 [Planner Agent] 계획 수립 중...")
    print("⚙️ [Worker Agent] 계획 실행 중...")
    print("🔍 [Reflection Agent] 결과 검토 중...")
    
    final_state = app_reflection.invoke(initial_state)
    
    print(f"\n✅ 실행 완료 (Reflection 횟수: {final_state['reflection_count']})\n")
    
    return final_state

# 테스트 실행
reflection_result = run_multi_agent_reflection("서울 2박3일 여행 계획 + 예산 50만원 이내")

print("=" * 50)
print("📊 Reflection 최종 결과")
print("=" * 50)
print(reflection_result["final_result"]) 

🚀 Reflection Multi-Agent 시스템 시작

📋 [Planner Agent] 계획 수립 중...
⚙️ [Worker Agent] 계획 실행 중...
🔍 [Reflection Agent] 결과 검토 중...

✅ 실행 완료 (Reflection 횟수: 1)

📊 Reflection 최종 결과
We need to produce a review of the worker's output, evaluating against the original request: "서울 2박3일 여행 계획 + 예산 50만원 이내". The worker provided a detailed plan with steps, budget allocation, itinerary, reservations, etc. Need to check if all requirements satisfied, missing info, accuracy, specificity, improvements needed, and then produce final improved result.

We should output in Korean as per original language. Provide a structured review.

Let's examine:

Original request: 2-night 3-day travel plan to Seoul, budget under 5 million KRW (500,000). Requirements: likely include itinerary, budget breakdown, accommodation, transportation, attractions, meals, contingency, etc. The worker gave a plan that includes all that.

Check: Does it meet all requirements? It includes 2 nights (Day1 and Day2 nights; Day3 morning depa